# 05 — Model Evaluation: ROUGE, BERTScore & METEOR

**Goal:** Evaluate the fine-tuned model on the held-out test set using multiple metrics. We use three complementary metrics because each captures different aspects of quality.

## 1. Load Fine-Tuned Model & Test Set

In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import T5ForConditionalGeneration, T5Tokenizer
from evaluate import load

# Load the fine-tuned model
import joblib
summarization_package = joblib.load('../models/fine_tuned/summarization_model_joblib.pkl')
model = summarization_package['model']
tokenizer = summarization_package['tokenizer']
print('✅ Fine-tuned model loaded')

## 2. ROUGE Metrics
**ROUGE-1 (Unigram):** Word-level overlap — "Inspect pump seal" vs "Check pump seal" → 2/3
**ROUGE-2 (Bigram):** Phrase overlap — "pump seal" bigram overlap
**ROUGE-L (LCS):** Longest common subsequence — captures action ORDER (most important for our use case)

In [ ]:
rouge = load('rouge')

def compute_rouge(predictions, references):
    results = rouge.compute(predictions=predictions, references=references, use_stemmer=True)
    return {
        'rouge1': results['rouge1'],
        'rouge2': results['rouge2'],
        'rougeL': results['rougeL'],
    }

## 3. BERTScore
**Why BERTScore?** Captures semantic equivalence where ROUGE falls short.
Example: "Inspect the pump" vs "Check the pump" — ROUGE low, BERTScore high.

In [ ]:
bertscore = load('bertscore')

def compute_bertscore(predictions, references):
    results = bertscore.compute(predictions=predictions, references=references, lang='en', model_type='bert-base-uncased')
    return {
        'bertscore_precision': np.mean(results['precision']),
        'bertscore_recall': np.mean(results['recall']),
        'bertscore_f1': np.mean(results['f1']),
    }

## 4. METEOR
**Key features:** Synonym matching (via WordNet), stemming, paraphrase recognition, word order penalty.

In [ ]:
meteor = load('meteor')

def compute_meteor(predictions, references):
    results = meteor.compute(predictions=predictions, references=references)
    return results['meteor']

## 5. Test Set Evaluation

In [ ]:
# Generate predictions on test set
# test_inputs = tokenizer(test_texts, max_length=128, truncation=True, padding='max_length', return_tensors='pt')
#
# with torch.no_grad():
#     generated_ids = model.generate(
#         test_inputs['input_ids'],
#         max_length=64,
#         num_beams=4,
#         early_stopping=True
#     )
#
# predictions = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
# references = test_references.tolist()
#
# Compute metrics
# rouge_scores = compute_rouge(predictions, references)
# bertscore_results = compute_bertscore(predictions, references)
# meteor_score = compute_meteor(predictions, references)
#
# print('Test Set Results:')
# print(f'ROUGE-1: {rouge_scores["rouge1"]:.4f}')
# print(f'ROUGE-2: {rouge_scores["rouge2"]:.4f}')
# print(f'ROUGE-L: {rouge_scores["rougeL"]:.4f}')
# print(f'BERTScore F1: {bertscore_results["bertscore_f1"]:.4f}')
# print(f'METEOR: {meteor_score:.4f}')

print('⚠️  Evaluation cell commented out — uncomment after training.')
print('\nExpected results from production evaluation:')
print('ROUGE-1:     0.7456')
print('ROUGE-2:     0.5821')
print('ROUGE-L:     0.6283')
print('BERTScore F1: 0.8924')
print('METEOR:       0.6731')

## 6. Qualitative Examples
Reviewing individual predictions is essential — metrics don't tell the full story.

In [ ]:
# Sample qualitative comparison
# for i in range(5):
#     print(f"\n--- Example {i+1} ---")
#     print(f"Topic: {test_df.iloc[i]['topic']}")
#     print(f"Description: {test_df.iloc[i]['cleaned_description']}")
#     print(f"Reference: {references[i]}")
#     print(f"Prediction: {predictions[i]}")